In [125]:
DATA_DIR = "../data"

In [132]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
import pandas as pd

# ============================================
# 1. Custom Dataset for DataFrame-based inputs
# ============================================

class ImageDFDataset(Dataset):
    def __init__(self, df, label_to_idx, transform=None):
        self.df = df.reset_index(drop=True)
        self.label_to_idx = label_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = DATA_DIR + '/' + self.df.loc[idx, "image_path"]
        label_str = self.df.loc[idx, "label"]
        label = self.label_to_idx[label_str]
        attributes = self.df.loc[idx, "attributes"]

        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img,attributes


# ============================================
# 2. Build label mapping
# ============================================

def build_label_mapping(df_train):
    classes = sorted(df_train["label"].unique())
    label_to_idx = {c: i for i, c in enumerate(classes)}
    idx_to_label = {i: c for c, i in label_to_idx.items()}
    return label_to_idx, idx_to_label


# ============================================
# 3. Transforms
# ============================================
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(128, scale=(0.8, 1.0)),  # Less aggressive crop
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),         # Milder colors
    transforms.RandomRotation(10),                         # Smaller rotation
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


val_transform = transforms.Compose([
    transforms.Resize(144),
    transforms.CenterCrop(128),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


# ============================================
# 4. Create datasets & loaders from dataframe
# ============================================

def make_dataloaders(df_train, df_val, batch_size=32):

    label_to_idx, idx_to_label = build_label_mapping(df_train)
    num_classes = len(label_to_idx)

    train_dataset = ImageDFDataset(df_train, label_to_idx, transform=train_transform)
    val_dataset   = ImageDFDataset(df_val, label_to_idx, transform=val_transform)

    # ---- Balanced sampler (important for ~20 images/class) ----
    class_counts = df_train["label"].value_counts().sort_index()
    class_weights = 1.0 / torch.tensor(class_counts.tolist(), dtype=torch.float)

    sample_weights = [
        class_weights[label_to_idx[label]]
        for label in df_train["label"]
    ]
    print(len(sample_weights))

    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler)
    val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, num_classes, label_to_idx, idx_to_label


# ============================================
# 5. Build ResNet-18 (train from scratch)
# ============================================

def build_resnet18(num_attributes):
    model = models.resnet18(weights=None)   # NOT pretrained

    model.fc = nn.Sequential(
        nn.Linear(512, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(0.4),
        nn.Linear(256, num_attributes),
    )
    return model


# ============================================
# 6. Training loop (clean version)
# ============================================

def train_model(model, train_loader, val_loader, epochs=100, lr=1e-4):

    criterion = nn.MSELoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    # backbone_params = [p for n, p in model.named_parameters() if not n.startswith("fc")]
    # head_params = model.fc.parameters()
    # optimizer = optim.AdamW([
    #     {'params': backbone_params, 'lr': 1e-5},  # 3x lower
    #     {'params': head_params,     'lr': 1e-3}
    # ], weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    for epoch in range(epochs):
        model.train()
        running_loss = 0


        for imgs, attributes in train_loader:
            imgs, attributes = imgs.to(device), attributes.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, attributes)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            running_loss += loss.item()

        val_loss = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        print(f"Epoch {epoch+1}/{epochs} "
              f"| Train Loss: {running_loss/len(train_loader):.4f} "
              f"| Val Loss: {val_loss:.4f}")


# ============================================
# 7. Validation
# ============================================

@torch.inference_mode()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0

    for imgs, att in loader:
        imgs, att = imgs.to(device), att.to(device)
        outputs = model(imgs)
        loss = criterion(outputs, att)
        total_loss += loss.item()

    return total_loss / len(loader)

In [133]:
def build_small_cnn(num_attributes, img_size=128):
    return nn.Sequential(
        nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.4),
        nn.Linear(128, num_attributes)
    )

In [134]:
import numpy as np
attributes = np.load(f"{DATA_DIR}/attributes.npy", allow_pickle=True)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
attributes_scaled = scaler.fit_transform(attributes)
from sklearn.decomposition import PCA

# Keep 2 principal components
pca = PCA(n_components=60)
attributes_pca = pca.fit_transform(attributes_scaled)
pca_std = attributes_pca.std(axis=0)  # Per-dimension stds
attributes_pca_normalized = attributes_pca / pca_std  # Now std=1 per dim
print("Explained variance ratio:", pca.explained_variance_ratio_.sum())

classes = [i + 1 for i in range(200)]

class_attributes_df = pd.DataFrame(
    columns=["label", "attributes"],
    data={
        "label": classes,
        "attributes": list(attributes_pca_normalized)
    }
)

Explained variance ratio: 0.9072184660778271


In [144]:
def build_tiny_cnn(num_attributes=60):
    return nn.Sequential(
        # Block 1: 3->16, downsample to 64x64
        nn.Conv2d(3, 16, 5, stride=2, padding=2),  # 128->64
        nn.ReLU(inplace=True),
        nn.BatchNorm2d(16),

        # Block 2: 16->32, downsample to 32x32
        nn.Conv2d(16, 32, 5, stride=2, padding=2), # 64->32
        nn.ReLU(inplace=True),
        nn.BatchNorm2d(32),

        # Global average pool + flatten
        nn.AdaptiveAvgPool2d((8, 8)),  # 32->8
        nn.Flatten(),

        # Head: 32*8*8=2048 -> 60
        nn.Linear(32 * 8 * 8, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(0.3),
        nn.Linear(256, num_attributes)
    )

In [150]:
from sklearn.model_selection import train_test_split
# ============================================
# 8. Example usage
# ============================================
df = pd.read_csv(f"{DATA_DIR}/train_images.csv")
df = df.merge(class_attributes_df)
df['attributes'] = [torch.tensor(arr, dtype=torch.float32) for arr in df['attributes']]

In [151]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

train_loader, val_loader, num_classes, label_to_idx, idx_to_label = \
    make_dataloaders(train_df, val_df)

model = build_micro_cnn(60)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

3140


This resolves the scaling issue
Your targets are now properly normalized (unit variance per dimension), so:

Initial random predictions → MSE ≈ 1.0 (reasonable starting point)

Converged model → MSE < 0.2 (good) or < 0.1 (excellent)

Gradients are now properly scaled for AdamW to work

In [152]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Params: {total_params:,}")  # Should be <100k for small data

Params: 13,356


In [153]:
train_model(model, train_loader, val_loader, epochs=10)

Epoch 1/10 | Train Loss: 1.0134 | Val Loss: 1.0713
Epoch 2/10 | Train Loss: 1.0166 | Val Loss: 1.0680
Epoch 3/10 | Train Loss: 0.9990 | Val Loss: 1.0661
Epoch 4/10 | Train Loss: 1.0155 | Val Loss: 1.0653
Epoch 5/10 | Train Loss: 0.9883 | Val Loss: 1.0646
Epoch 6/10 | Train Loss: 0.9989 | Val Loss: 1.0640
Epoch 7/10 | Train Loss: 1.0093 | Val Loss: 1.0639
Epoch 8/10 | Train Loss: 1.0064 | Val Loss: 1.0636
Epoch 9/10 | Train Loss: 0.9994 | Val Loss: 1.0635
Epoch 10/10 | Train Loss: 0.9902 | Val Loss: 1.0635


In [154]:
print("Unique attribute vectors:", train_df["attributes"].nunique())
print("Expected: ~200")
print("Attributes std across dataset:", np.stack(train_df["attributes"]).std())
print("Expected: ~1.0")


Unique attribute vectors: 3140
Expected: ~200
Attributes std across dataset: 1.0261742
Expected: ~1.0


Params: 13,356
